# LeetCode 1035 — Uncrossed Lines bằng Python

Link đề: https://leetcode.com/problems/uncrossed-lines/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Ghi chú: viết các số của `nums1` trên một đường ngang và các số của `nums2` trên một đường ngang khác (đúng thứ tự). Ta có thể nối `nums1[i]` với `nums2[j]` bằng một đoạn thẳng nếu **hai số bằng nhau** và đoạn thẳng **không cắt** đoạn nào khác (kể cả chạm tại đầu mút: mỗi số thuộc tối đa một đoạn). Tìm **số đoạn thẳng nhiều nhất**. Phát biểu tương đương (và đây là ý then chốt): tập các đoạn không cắt nhau chính là một **dãy con chung** (common subsequence) của hai mảng, nên đáp án là **độ dài dãy con chung dài nhất** (LCS, *Longest Common Subsequence*). Đây là bài quy hoạch động kinh điển `O(n * m)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và **chuyển đổi** bài toán hình học (đoạn thẳng không cắt) thành bài toán chuỗi (dãy con chung dài nhất).
2. Nắm các công cụ Python cần thiết: bảng 2 chiều `[[0] * (m + 1) for _ in range(n + 1)]`, `functools.lru_cache`, mảng cuộn (rolling array), `bisect_left` cho LIS, và `defaultdict(list)` cho danh sách vị trí.
3. Hiểu nguyên lý **quy hoạch động**: `dp[i][j]` là số đoạn nhiều nhất giữa `nums1[:i]` và `nums2[:j]`; nếu `nums1[i-1] == nums2[j-1]` thì `dp[i][j] = dp[i-1][j-1] + 1`, ngược lại `max(dp[i-1][j], dp[i][j-1])`.
4. Đi qua 5 cách: vét cạn tập các đoạn thẳng thỏa điều kiện (oracle theo đúng định nghĩa), đệ quy có nhớ, bảng 2 chiều, mảng cuộn một hàng (cách nộp chính) và cách dùng **LIS** (dãy con tăng dài nhất) trên danh sách vị trí.
5. Tránh các lỗi kinh điển: tham lam nối ngay các cặp bằng nhau, thiếu `max` của hai ô khi không khớp, nhầm với **chuỗi con liên tiếp** (substring), đếm giá trị chung (tập hợp) thay vì LCS, kích thước bảng lệch một đơn vị, và ghi đè ô chéo khi dùng mảng cuộn.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2`. Ta viết các số của `nums1` và `nums2` (theo đúng thứ tự cho trước) trên hai **đường ngang** riêng biệt.

Ta có thể vẽ các **đoạn thẳng nối**: nối `nums1[i]` với `nums2[j]` sao cho:

- `nums1[i] == nums2[j]`, và
- đoạn thẳng đó **không cắt** bất kỳ đoạn nối nào khác.

Lưu ý một đoạn nối không được cắt ngay cả tại đầu mút (mỗi số chỉ thuộc **một** đoạn nối).

Trả về **số đoạn nối nhiều nhất** có thể vẽ.

**Ví dụ 1:** `nums1 = [1,4,2]`, `nums2 = [1,2,4]` -> `2`. Không thể vẽ 3 đoạn vì đoạn nối `4` (vị trí 1 và vị trí 2) cắt đoạn nối `2` (vị trí 2 và vị trí 1).

**Ví dụ 2:** `nums1 = [2,5,1,2,5]`, `nums2 = [10,5,2,1,5,2]` -> `3`.

**Ví dụ 3:** `nums1 = [1,3,7,1,7,5]`, `nums2 = [1,9,2,5,1]` -> `2`.

**Ràng buộc:**

- `1 <= nums1.length, nums2.length <= 500`
- `1 <= nums1[i], nums2[j] <= 2000`

In [1]:
from typing import List, Tuple, Dict, Optional
from functools import lru_cache
from bisect import bisect_left
from collections import defaultdict
from itertools import product
import sys
import random, time

examples = [
    ([1, 4, 2], [1, 2, 4], 2),
    ([2, 5, 1, 2, 5], [10, 5, 2, 1, 5, 2], 3),
    ([1, 3, 7, 1, 7, 5], [1, 9, 2, 5, 1], 2),
]
for a, b, expected in examples:
    print(f"nums1={a} nums2={b} -> kỳ vọng {expected}")

nums1=[1, 4, 2] nums2=[1, 2, 4] -> kỳ vọng 2
nums1=[2, 5, 1, 2, 5] nums2=[10, 5, 2, 1, 5, 2] -> kỳ vọng 3
nums1=[1, 3, 7, 1, 7, 5] nums2=[1, 9, 2, 5, 1] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Hai đoạn nối có cắt nhau không?

Đoạn nối `(i1, j1)` và `(i2, j2)` (nối vị trí `i` của `nums1` với vị trí `j` của `nums2`). Hai đoạn **không cắt và không chung đầu mút** ⇔ `(i1 - i2) * (j1 - j2) > 0`: cả hai chỉ số cùng tăng (hoặc cùng giảm). Nếu tích `< 0` thì hai đoạn **cắt nhau**; nếu `= 0` thì chúng **chung một đầu mút** (không được phép).

In [2]:
def compatible(l1, l2):
    return (l1[0] - l2[0]) * (l1[1] - l2[1]) > 0

print("(0,0) và (2,1):", compatible((0, 0), (2, 1)), "| (1,2) và (2,1) cắt nhau:", compatible((1, 2), (2, 1)), "| chung đầu mút (1,1) và (1,3):", compatible((1, 1), (1, 3)))

(0,0) và (2,1): True | (1,2) và (2,1) cắt nhau: False | chung đầu mút (1,1) và (1,3): False


### 3.2. Tập đoạn nối hợp lệ = dãy con chung

Sắp xếp các đoạn nối của một tập hợp hợp lệ theo `i` tăng dần thì `j` cũng tăng ngặt. Các giá trị `nums1[i_k] = nums2[j_k]` tạo thành một dãy (theo thứ tự) xuất hiện cả trong `nums1` (tại các vị trí `i_k` tăng) lẫn trong `nums2` (tại `j_k` tăng): đó là một **dãy con chung** (subsequence, không cần liên tiếp). Ngược lại mọi dãy con chung cho một tập đoạn nối hợp lệ. Vậy số đoạn nhiều nhất = độ dài dãy con chung dài nhất (**LCS**).

### 3.3. Bảng quy hoạch động 2 chiều

Gọi `dp[i][j]` = LCS của `nums1[:i]` và `nums2[:j]` (`i` phần tử đầu và `j` phần tử đầu). Bảng có kích thước `(n + 1) x (m + 1)` với hàng/cột `0` là `0` (một mảng rỗng). Điền từ trên xuống, từ trái sang phải: mỗi ô phụ thuộc ba ô đã tính: trên, trái và chéo.

In [3]:
a, b = [1, 4, 2], [1, 2, 4]
n, m = len(a), len(b)
dp = [[0] * (m + 1) for _ in range(n + 1)]
for i in range(1, n + 1):
    for j in range(1, m + 1):
        dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
print("cột:   ", [""] + b)
for i, row in enumerate(dp):
    print(f"{'' if i == 0 else a[i - 1]!s:>4}", row)
print("đáp án = dp[n][m] =", dp[n][m])
assert dp[n][m] == 2

cột:    ['', 1, 2, 4]
     [0, 0, 0, 0]
   1 [0, 1, 1, 1]
   4 [0, 1, 1, 2]
   2 [0, 1, 2, 2]
đáp án = dp[n][m] = 2


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `nums1 = [1, 4, 2]`, `nums2 = [1, 2, 4]`.

Các cặp có thể nối: `1-1` (vị trí `(0, 0)`), `4-4` (vị trí `(1, 2)`), `2-2` (vị trí `(2, 1)`). Hai cặp `(1, 2)` và `(2, 1)` **cắt nhau** (`(1 - 2) * (2 - 1) < 0`), nên không thể dùng đồng thời; hai cặp còn lại với `(0, 0)` đều tương thích. Ta chọn `(0, 0)` và một trong hai cặp kia: **2** đoạn. (Dãy con chung là `[1, 4]` hoặc `[1, 2]`.)

Đây chính là LCS của `[1, 4, 2]` và `[1, 2, 4]`: `[1, 2]` hoặc `[1, 4]`, độ dài `2`.

## 5. Cách 1 — Brute Force: thử mọi tập đoạn nối theo đúng định nghĩa

**Ý tưởng.** Liệt kê các **đoạn nối ứng viên** `(i, j)` với `nums1[i] == nums2[j]`. Duyệt (quay lui) mọi tập con các ứng viên mà **mọi cặp** trong tập đều tương thích (`(i1 - i2) * (j1 - j2) > 0`); lấy tập lớn nhất. Không dùng quy hoạch động hay LCS nên đây là **oracle độc lập** (làm đúng theo phát biểu hình học của đề). Chỉ dùng cho mảng ngắn.

In [4]:
def lines_brute(nums1: List[int], nums2: List[int]) -> int:
    cand = [(i, j) for i, x in enumerate(nums1) for j, y in enumerate(nums2) if x == y]
    best = 0

    def rec(start: int, chosen: List[Tuple[int, int]]) -> None:
        nonlocal best
        best = max(best, len(chosen))
        for idx in range(start, len(cand)):
            line = cand[idx]
            if all(compatible(line, other) for other in chosen):
                chosen.append(line)
                rec(idx + 1, chosen)
                chosen.pop()

    rec(0, [])
    return best

for a, b, expected in examples:
    got = lines_brute(a, b)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 2 OK
brute = 3 OK
brute = 2 OK


**Độ phức tạp và nút thắt.** Số tập đoạn nối hợp lệ tăng rất nhanh (hai mảng toàn số giống nhau độ dài `k` có `C(2k, k)` tập). Đo thử:

In [5]:
for k in (3, 5, 7, 8):
    a = b = [1] * k
    t0 = time.perf_counter()
    r = lines_brute(a, b)
    print(f"toàn số giống nhau, độ dài {k}: đáp án {r}, {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("độ dài 500: không thể")

toàn số giống nhau, độ dài 3: đáp án 3,       0.1 ms
toàn số giống nhau, độ dài 5: đáp án 5,       1.0 ms
toàn số giống nhau, độ dài 7: đáp án 7,      20.8 ms
toàn số giống nhau, độ dài 8: đáp án 8,      49.1 ms
độ dài 500: không thể


Nút thắt: ta xét lại cùng các tiền tố nhiều lần. **Bài toán con** "số đoạn nhiều nhất giữa `nums1[:i]` và `nums2[:j]`" lặp lại: dùng quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Quy hoạch động theo tiền tố.** Xét hai phần tử cuối `nums1[i-1]` và `nums2[j-1]`:

- Nếu **bằng nhau**: ta có thể nối chúng. Không mất tổng quát khi nối (trao đổi), phần còn lại là bài toán `(i-1, j-1)`: `dp[i][j] = dp[i-1][j-1] + 1`.
- Nếu **khác nhau**: ít nhất một trong hai không được nối (không thể nối `nums1[i-1]` với `nums2[j-1]`; nếu cả hai đều được nối thì với số khác ở đầu kia, mà hai đầu cuối đồng thời nối được với nhau chỉ khi bằng). Bỏ `nums1[i-1]` hoặc bỏ `nums2[j-1]`: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Đáp án `dp[n][m]`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy có nhớ theo `(i, j)` |
| 3 | Bảng 2 chiều bottom-up |
| 4 | Mảng cuộn một hàng, `O(min(n, m))` bộ nhớ (nộp) |
| 5 | LCS qua LIS: danh sách vị trí và `bisect` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích từng đoạn.**

- `f(i, j)` = LCS của `nums1[i:]` và `nums2[j:]` (xét từ đầu về cuối).
- `i == n` hoặc `j == m`: `0`.
- Bằng nhau: `1 + f(i + 1, j + 1)`; khác nhau: `max(f(i + 1, j), f(i, j + 1))`.
- Độ sâu đệ quy tới `n + m = 1000` nên cần nâng giới hạn đệ quy (`sys.setrecursionlimit`).

In [6]:
def lines_memo(nums1: List[int], nums2: List[int]) -> int:
    n, m = len(nums1), len(nums2)
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 5000))

    @lru_cache(None)
    def f(i: int, j: int) -> int:
        if i == n or j == m:
            return 0
        if nums1[i] == nums2[j]:
            return 1 + f(i + 1, j + 1)
        return max(f(i + 1, j), f(i, j + 1))

    return f(0, 0)

for a, b, expected in examples:
    got = lines_memo(a, b)
    print(f"đệ quy có nhớ = {got}", "OK" if got == expected else "SAI")
    assert got == expected

đệ quy có nhớ = 2 OK
đệ quy có nhớ = 3 OK
đệ quy có nhớ = 2 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        import sys
        from functools import lru_cache
        sys.setrecursionlimit(10000)
        n, m = len(nums1), len(nums2)

        @lru_cache(None)
        def f(i: int, j: int) -> int:
            if i == n or j == m:
                return 0
            if nums1[i] == nums2[j]:
                return 1 + f(i + 1, j + 1)
            return max(f(i + 1, j), f(i, j + 1))

        return f(0, 0)

sol_memo = Solution().maxUncrossedLines
for a, b, expected in examples:
    assert sol_memo(a, b) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n * m)` trạng thái, mỗi trạng thái `O(1)`; bộ nhớ `O(n * m)` cho bảng nhớ và `O(n + m)` ngăn xếp đệ quy.

---

### 7.2. Cách 3 — Bảng 2 chiều bottom-up

**Giải thích từng đoạn.** Điền `dp` như ở mục 3.3: `dp[i][j]` dùng ô chéo `dp[i-1][j-1]` khi hai số bằng nhau, hoặc lấy `max` của ô trên và ô bên trái khi khác nhau. Đáp án `dp[n][m]`.

In [8]:
def lines_table(nums1: List[int], nums2: List[int]) -> int:
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if nums1[i - 1] == nums2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a, b, expected in examples:
    got = lines_table(a, b)
    print(f"bảng 2 chiều = {got}", "OK" if got == expected else "SAI")
    assert got == expected

bảng 2 chiều = 2 OK
bảng 2 chiều = 3 OK
bảng 2 chiều = 2 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        n, m = len(nums1), len(nums2)
        dp = [[0] * (m + 1) for _ in range(n + 1)]       # dp[i][j]: LCS của nums1[:i] và nums2[:j]
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                if nums1[i - 1] == nums2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[n][m]

sol_table = Solution().maxUncrossedLines
for a, b, expected in examples:
    assert sol_table(a, b) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n * m)` (`250000` ô khi `n = m = 500`), bộ nhớ `O(n * m)`.

---

### 7.3. Cách 4 — Mảng cuộn một hàng (cách nộp chính)

**Ý tưởng khác.** Mỗi hàng `dp[i]` chỉ phụ thuộc hàng `dp[i-1]`, nên chỉ cần **một mảng** độ dài `m + 1` (đặt `m` là độ dài mảng ngắn hơn để tiết kiệm). Khi cập nhật tại chỗ, giá trị chéo `dp[i-1][j-1]` sẽ bị ghi đè; ta **cất** nó vào biến `prev` trước khi ghi.

**Giải thích từng đoạn.**

- `dp[j]` đang giữ `dp[i-1][j]` (ô trên); khi cập nhật `dp[j]` thành `dp[i][j]`, ô trái `dp[j-1]` đã là `dp[i][j-1]`.
- `prev` giữ `dp[i-1][j-1]` (ô chéo): trước khi ghi `dp[j]`, lưu `tmp = dp[j]` (ô trên cũ) để dùng làm ô chéo cho `j + 1`.
- Nếu `nums1[i-1] == nums2[j-1]`: `dp[j] = prev + 1`, ngược lại `dp[j] = max(dp[j], dp[j-1])`.

In [10]:
def lines_rolling(nums1: List[int], nums2: List[int]) -> int:
    if len(nums2) > len(nums1):                  # cho mảng ngắn hơn làm chiều cột để tiết kiệm bộ nhớ
        nums1, nums2 = nums2, nums1
    m = len(nums2)
    dp = [0] * (m + 1)
    for x in nums1:
        prev = 0                                 # dp[i-1][j-1]
        for j in range(1, m + 1):
            tmp = dp[j]                          # dp[i-1][j]: sẽ là ô chéo của j + 1
            if x == nums2[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

for a, b, expected in examples:
    got = lines_rolling(a, b)
    print(f"mảng cuộn = {got}", "OK" if got == expected else "SAI")
    assert got == expected

mảng cuộn = 2 OK
mảng cuộn = 3 OK
mảng cuộn = 2 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        if len(nums2) > len(nums1):              # chiều cột là mảng ngắn hơn
            nums1, nums2 = nums2, nums1
        m = len(nums2)
        dp = [0] * (m + 1)                       # dp[j]: LCS của tiền tố nums1 hiện tại và nums2[:j]
        for x in nums1:
            prev = 0                             # ô chéo dp[i-1][j-1]
            for j in range(1, m + 1):
                tmp = dp[j]                      # ô trên cũ dp[i-1][j] -> thành ô chéo cho j + 1
                if x == nums2[j - 1]:
                    dp[j] = prev + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return dp[m]

sol_rolling = Solution().maxUncrossedLines
for a, b, expected in examples:
    assert sol_rolling(a, b) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n * m)`, bộ nhớ `O(min(n, m))`.

---

### 7.4. Cách 5 — LCS qua LIS (Hunt-Szymanski)

**Ý tưởng khác.** LCS bằng độ dài **dãy con tăng ngặt dài nhất** (LIS) của một dãy chỉ số được dựng như sau: với mỗi số `x` của `nums1` (theo thứ tự), liệt kê các vị trí của `x` trong `nums2` theo thứ tự **giảm dần** rồi nối vào dãy `seq`. Một dãy con tăng ngặt của `seq` tương ứng một tập đoạn nối `(i, j)` có `i` tăng (vì thuộc các `x` khác nhau theo thứ tự) và `j` tăng ngặt; chiều giảm dần trong từng nhóm đảm bảo mỗi `x` (mỗi `i`) dùng **tối đa một** vị trí `j`. LIS tính bằng `bisect_left` trên mảng đuôi (patience sorting) trong `O(L log L)` với `L = |seq|`.

**Giải thích từng đoạn.**

- `pos[v]`: danh sách vị trí tăng dần của giá trị `v` trong `nums2`.
- `tails`: `tails[k]` là chỉ số `j` nhỏ nhất kết thúc một dãy con tăng ngặt độ dài `k + 1`.
- Với mỗi `j` trong `seq`: `k = bisect_left(tails, j)`; nếu `k == len(tails)` thì thêm, ngược lại thay.
- Đáp án `len(tails)`.

In [12]:
def lines_lis(nums1: List[int], nums2: List[int]) -> int:
    pos: Dict[int, List[int]] = defaultdict(list)
    for j, y in enumerate(nums2):
        pos[y].append(j)
    tails: List[int] = []
    for x in nums1:
        for j in reversed(pos.get(x, ())):           # giảm dần: mỗi x chỉ dùng tối đa một vị trí
            k = bisect_left(tails, j)
            if k == len(tails):
                tails.append(j)
            else:
                tails[k] = j
    return len(tails)

for a, b, expected in examples:
    got = lines_lis(a, b)
    print(f"LIS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

LIS = 2 OK
LIS = 3 OK
LIS = 2 OK


**Code nộp LeetCode (Cách 5):**

In [13]:
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        from bisect import bisect_left
        from collections import defaultdict
        pos = defaultdict(list)                   # giá trị -> các vị trí (tăng dần) trong nums2
        for j, y in enumerate(nums2):
            pos[y].append(j)
        tails = []                                # LIS tăng ngặt trên dãy vị trí
        for x in nums1:
            for j in reversed(pos.get(x, ())):    # giảm dần để mỗi x dùng tối đa một vị trí
                k = bisect_left(tails, j)
                if k == len(tails):
                    tails.append(j)
                else:
                    tails[k] = j
        return len(tails)

sol_lis = Solution().maxUncrossedLines
for a, b, expected in examples:
    assert sol_lis(a, b) == expected
print("Cách 5 qua ví dụ")

Cách 5 qua ví dụ


**Độ phức tạp (Cách 5):** thời gian `O(R log R)` với `R` là số cặp `(i, j)` bằng nhau (tối đa `n * m = 250000` khi mọi số giống nhau, nhưng thường rất nhỏ khi giá trị đa dạng); bộ nhớ `O(m)`.

## 8. TRACE từng bước

### 8.1. Bảng quy hoạch động cho ví dụ 2

`nums1 = [2, 5, 1, 2, 5]`, `nums2 = [10, 5, 2, 1, 5, 2]`: in toàn bộ bảng và tô dấu ô lấy từ ô chéo.

In [14]:
def trace_table(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
    print("       " + " ".join(f"{v:>3}" for v in [""] + b))
    for i, row in enumerate(dp):
        print(f"{'' if i == 0 else a[i - 1]!s:>4}   " + " ".join(f"{v:>3}" for v in row))
    return dp[n][m]

assert trace_table([2, 5, 1, 2, 5], [10, 5, 2, 1, 5, 2]) == 3

            10   5   2   1   5   2
         0   0   0   0   0   0   0
   2     0   0   0   1   1   1   1
   5     0   0   1   1   1   2   2
   1     0   0   1   1   2   2   2
   2     0   0   1   2   2   2   3
   5     0   0   1   2   2   3   3


### 8.2. Mảng cuộn cho ví dụ 1

In `dp` sau mỗi phần tử của `nums1` (với `nums2 = [1, 2, 4]`).

In [15]:
nums1, nums2 = [1, 4, 2], [1, 2, 4]
dp = [0] * (len(nums2) + 1)
print("ban đầu:", dp)
for x in nums1:
    prev = 0
    for j in range(1, len(nums2) + 1):
        tmp = dp[j]
        dp[j] = prev + 1 if x == nums2[j - 1] else max(dp[j], dp[j - 1])
        prev = tmp
    print(f"sau x = {x}:", dp)
assert dp[-1] == 2

ban đầu: [0, 0, 0, 0]
sau x = 1: [0, 1, 1, 1]
sau x = 4: [0, 1, 1, 2]
sau x = 2: [0, 1, 2, 2]


### 8.3. LIS trên dãy vị trí cho ví dụ 2

In dãy `seq` (các vị trí theo thứ tự đã dựng) và mảng `tails` sau mỗi phần tử.

In [16]:
nums1, nums2 = [2, 5, 1, 2, 5], [10, 5, 2, 1, 5, 2]
pos = defaultdict(list)
for j, y in enumerate(nums2):
    pos[y].append(j)
seq = [j for x in nums1 for j in reversed(pos.get(x, ()))]
print("seq =", seq)
tails = []
for j in seq:
    k = bisect_left(tails, j)
    if k == len(tails):
        tails.append(j)
    else:
        tails[k] = j
    print(f"j={j}: tails = {tails}")
assert len(tails) == 3

seq = [5, 2, 4, 1, 3, 5, 2, 4, 1]
j=5: tails = [5]
j=2: tails = [2]
j=4: tails = [2, 4]
j=1: tails = [1, 4]
j=3: tails = [1, 3]
j=5: tails = [1, 3, 5]
j=2: tails = [1, 2, 5]
j=4: tails = [1, 2, 4]
j=1: tails = [1, 2, 4]


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (mảng cuộn một hàng): `O(n * m)` thời gian, `O(min(n, m))` bộ nhớ.

```python
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        if len(nums2) > len(nums1):              # chiều cột là mảng ngắn hơn
            nums1, nums2 = nums2, nums1
        m = len(nums2)
        dp = [0] * (m + 1)                       # dp[j]: LCS của tiền tố nums1 hiện tại và nums2[:j]
        for x in nums1:
            prev = 0                             # ô chéo dp[i-1][j-1]
            for j in range(1, m + 1):
                tmp = dp[j]                      # ô trên cũ dp[i-1][j] -> thành ô chéo cho j + 1
                if x == nums2[j - 1]:
                    dp[j] = prev + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return dp[m]
```

In [17]:
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        if len(nums2) > len(nums1):              # chiều cột là mảng ngắn hơn
            nums1, nums2 = nums2, nums1
        m = len(nums2)
        dp = [0] * (m + 1)                       # dp[j]: LCS của tiền tố nums1 hiện tại và nums2[:j]
        for x in nums1:
            prev = 0                             # ô chéo dp[i-1][j-1]
            for j in range(1, m + 1):
                tmp = dp[j]                      # ô trên cũ dp[i-1][j] -> thành ô chéo cho j + 1
                if x == nums2[j - 1]:
                    dp[j] = prev + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return dp[m]

sol = Solution()
for a, b, expected in examples:
    got = sol.maxUncrossedLines(a, b)
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

2 OK
3 OK
2 OK


## 10. Vì sao đúng

**Từ đoạn nối sang dãy con chung (mục 3.2).** Một tập đoạn nối hợp lệ có thể sắp xếp theo `i` tăng; khi đó `j` tăng ngặt (không cắt và không chung đầu mút) và `nums1[i_k] = nums2[j_k]`, nên các giá trị tạo một dãy con chung. Ngược lại một dãy con chung với các chỉ số tăng ngặt ở cả hai mảng cho một tập đoạn nối hợp lệ cùng kích thước. Vậy đáp án = LCS.

**Công thức quy hoạch động.** Gọi `L(i, j)` là LCS của `nums1[:i]` và `nums2[:j]`.

- Nếu `nums1[i-1] == nums2[j-1]`: tồn tại một LCS dùng cặp cuối này (trao đổi: nếu một LCS không dùng `nums1[i-1]` hoặc `nums2[j-1]` ở cặp cuối thì thay phần tử cuối của LCS bằng cặp `(i-1, j-1)` không giảm độ dài, vì cặp này nằm sau mọi chỉ số khác), nên `L(i, j) = L(i-1, j-1) + 1` (không thể lớn hơn vì `L(i, j) <= L(i-1, j-1) + 1`).
- Nếu khác nhau: trong một LCS tối ưu, `nums1[i-1]` và `nums2[j-1]` không được cùng nằm ở cặp cuối (chúng khác nhau), nên một trong hai không xuất hiện: `L(i, j) = max(L(i-1, j), L(i, j-1))`. Bất đẳng thức ngược `L(i, j) >= max(...)` hiển nhiên.

Điều kiện biên `L(0, j) = L(i, 0) = 0`. Bảng `(n + 1) x (m + 1)` điền theo thứ tự tăng của `(i, j)` nên mọi ô cần đã sẵn.

**Mảng cuộn.** Hàng `i` chỉ cần hàng `i - 1`; biến `prev` giữ ô chéo trước khi bị ghi đè nên cập nhật tại chỗ là đúng.

**Cách 5 (LIS).** Tập đoạn nối hợp lệ ⇔ dãy cặp `(i_k, j_k)` có `i_k` tăng ngặt và `j_k` tăng ngặt. Dãy `seq` liệt kê, theo thứ tự `i` tăng, các `j` của từng `i` theo thứ tự **giảm** dần. Một dãy con tăng **ngặt** của `seq` chọn được tối đa một `j` cho mỗi `i` (hai `j` cùng nhóm đứng theo thứ tự giảm nên không thể cùng nằm trong dãy tăng ngặt) và các nhóm theo thứ tự `i` tăng; đó chính là tập đoạn nối hợp lệ. Vậy LCS = độ dài LIS ngặt của `seq`, tính bằng `bisect_left`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) duyệt mọi tập đoạn nối thỏa điều kiện không cắt theo đúng phát biểu hình học. Ta đối chiếu bốn cách còn lại trên **mọi** cặp mảng có giá trị trong `{1, 2, 3}` và độ dài `<= 4` (cả hai mảng), cộng nhiều ca ngẫu nhiên có seed (cỡ `500 x 500`, so các cách nhanh với nhau).

In [18]:
total = 0
for la in range(1, 5):
    for lb in range(1, 5):
        for ta in product((1, 2, 3), repeat=la):
            a = list(ta)
            for tb in product((1, 2, 3), repeat=lb):
                b = list(tb)
                ref = lines_brute(a, b)
                assert lines_memo(a, b) == lines_table(a, b) == lines_rolling(a, b) == lines_lis(a, b) == ref, (a, b)
                assert sol.maxUncrossedLines(a, b) == ref
                total += 1
print("đã đối chiếu", total, "cặp mảng (giá trị 1..3, độ dài 1..4 mỗi mảng): các cách khớp oracle hình học")

đã đối chiếu 14400 cặp mảng (giá trị 1..3, độ dài 1..4 mỗi mảng): các cách khớp oracle hình học


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập đoạn nối | hàm mũ | `O(n + m)` |
| 2. Đệ quy có nhớ | `O(n * m)` | `O(n * m)` |
| 3. Bảng 2 chiều | `O(n * m)` | `O(n * m)` |
| 4. Mảng cuộn | `O(n * m)` | `O(min(n, m))` |
| 5. LIS qua vị trí | `O(R log R)`, `R` số cặp bằng nhau | `O(m)` |

Đo ở cỡ tối đa (`n = m = 500`) với các dạng: giá trị ngẫu nhiên thưa (`1..2000`), ít giá trị khác nhau (`1..3`), toàn số giống nhau, hai mảng giống hệt, và hai mảng không có giá trị chung:

In [19]:
rng = random.Random(11)
n = m = 500
cases = {
    "ngẫu nhiên 1..2000": ([rng.randint(1, 2000) for _ in range(n)], [rng.randint(1, 2000) for _ in range(m)]),
    "ngẫu nhiên 1..3": ([rng.randint(1, 3) for _ in range(n)], [rng.randint(1, 3) for _ in range(m)]),
    "toàn số giống nhau": ([7] * n, [7] * m),
    "hai mảng giống hệt": (list(range(1, n + 1)), list(range(1, n + 1))),
    "không có giá trị chung": ([1] * n, [2] * m),
}
for label, (a, b) in cases.items():
    ref = lines_table(a, b)
    row = [f"{label:24}"]
    for name, fn in (("2D", sol_table), ("cuộn", sol_rolling), ("LIS", sol_lis), ("nộp", sol.maxUncrossedLines)):
        t0 = time.perf_counter()
        got = fn(a, b)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.1f} ms")
    t0 = time.perf_counter()
    got = sol_memo(a, b)
    row.append(f"đệ quy {(time.perf_counter() - t0) * 1000:7.1f} ms")
    assert got == ref
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên 1..2000       | 2D    34.0 ms | cuộn    21.8 ms | LIS     0.2 ms | nộp    22.2 ms | đệ quy   154.0 ms | đáp án 20
ngẫu nhiên 1..3          | 2D    30.3 ms | cuộn    24.7 ms | LIS    10.2 ms | nộp    22.8 ms | đệ quy    76.9 ms | đáp án 361


toàn số giống nhau       | 2D    25.7 ms | cuộn    29.6 ms | LIS    44.8 ms | nộp    23.3 ms | đệ quy     0.4 ms | đáp án 500
hai mảng giống hệt       | 2D    27.6 ms | cuộn    21.1 ms | LIS     0.2 ms | nộp    21.2 ms | đệ quy     0.4 ms | đáp án 500


không có giá trị chung   | 2D    27.7 ms | cuộn    21.7 ms | LIS     0.1 ms | nộp    22.0 ms | đệ quy   145.1 ms | đáp án 0


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc, cả `500 x 500`) và các tính chất.

In [20]:
def all_solutions():
    return {"memo": lines_memo, "table": lines_table, "rolling": lines_rolling, "lis": lines_lis,
            "sol_memo": sol_memo, "sol_table": sol_table, "sol_rolling": sol_rolling, "sol_lis": sol_lis, "final": sol.maxUncrossedLines}

def check_all(a, b, expected=None, brute=True):
    results = {name: fn(list(a), list(b)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (a[:10], b[:10], results)
    ref = results["table"]
    if brute and len(a) <= 7 and len(b) <= 7:
        assert lines_brute(a, b) == ref, (a, b)
    if expected is not None:
        assert ref == expected, (a, b, ref, expected)
    return ref

# 12.1 ví dụ đề
for a, b, expected in examples:
    check_all(a, b, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [1], 1),                                  # một cặp bằng nhau
    ([1], [2], 0),                                  # không có giá trị chung
    ([1, 2, 3], [1, 2, 3], 3),                      # hai mảng giống hệt: nối tất cả theo đường thẳng đứng
    ([1, 2, 3], [3, 2, 1], 1),                      # đảo ngược: mọi cặp cắt nhau, chỉ chọn một
    ([1, 1, 1], [1, 1], 2),                         # trùng lặp: giới hạn bởi mảng ngắn hơn
    ([1, 2, 1], [1], 1),                            # một đầu chỉ có một phần tử: tối đa 1
    ([5, 5, 5, 5], [5, 5, 5, 5, 5], 4),             # toàn số giống nhau
    ([1, 3, 5], [2, 4, 6], 0),                      # rời nhau
    ([1, 2, 3, 4], [2, 4], 2),                      # mảng ngắn là dãy con của mảng dài
    ([1, 2, 3, 4], [4, 3, 2, 1], 1),                # đảo ngược hoàn toàn
    ([1, 2, 1, 2, 1], [2, 1, 2, 1, 2], 4),          # xen kẽ: dãy con chung 2,1,2,1 hoặc 1,2,1,2
    ([1, 3, 2, 1], [3, 1, 2], 2),                   # tính tay ở bài tập 2
    ([2000, 1, 2000], [2000, 2000], 2),             # giá trị biên 2000
    ([1] * 500, [1] * 500, 500),                    # cỡ tối đa: toàn số giống nhau
    (list(range(1, 501)), list(range(500, 0, -1)), 1),   # cỡ tối đa: một dãy tăng và một dãy giảm
]
for a, b, expected in edge:
    check_all(a, b, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 15 ca


In [21]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle hình học)
def wrong_greedy(nums1, nums2):                    # tham lam: lần lượt nối mỗi nums1[i] với vị trí khớp đầu tiên sau đoạn nối trước
    j, count = 0, 0
    for x in nums1:
        for k in range(j, len(nums2)):
            if nums2[k] == x:
                count += 1
                j = k + 1
                break
    return count

def wrong_no_max(nums1, nums2):                    # khi khác nhau chỉ lấy dp[i-1][j-1] (thiếu max của hai ô)
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if nums1[i - 1] == nums2[j - 1] else dp[i - 1][j - 1]
    return dp[n][m]

def wrong_substring(nums1, nums2):                 # nhầm với chuỗi con liên tiếp: reset về 0 khi khác nhau
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    best = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if nums1[i - 1] == nums2[j - 1] else 0
            best = max(best, dp[i][j])
    return best

def wrong_set_intersection(nums1, nums2):          # đếm số giá trị chung khác nhau thay vì LCS
    return len(set(nums1) & set(nums2))

def wrong_rolling_no_prev(nums1, nums2):           # mảng cuộn: quên cất ô chéo (dùng dp[j-1] cũ thay vì ô chéo thật)
    m = len(nums2)
    dp = [0] * (m + 1)
    for x in nums1:
        for j in range(1, m + 1):
            if x == nums2[j - 1]:
                dp[j] = dp[j - 1] + 1                      # sai: dp[j - 1] đã là hàng mới, không phải ô chéo
            else:
                dp[j] = max(dp[j], dp[j - 1])
    return dp[m]

def find_counter(wrong, vmax=3, max_len=5):
    for la in range(1, max_len + 1):
        for lb in range(1, max_len + 1):
            for ta in product(range(1, vmax + 1), repeat=la):
                for tb in product(range(1, vmax + 1), repeat=lb):
                    if wrong(list(ta), list(tb)) != lines_brute(list(ta), list(tb)):
                        return list(ta), list(tb)
    return None

counters = {}
for name, fn in (("tham lam nối ngay", wrong_greedy), ("thiếu max khi khác nhau", wrong_no_max),
                 ("nhầm với chuỗi con liên tiếp", wrong_substring), ("đếm giá trị chung (tập hợp)", wrong_set_intersection),
                 ("mảng cuộn quên ô chéo", wrong_rolling_no_prev)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    a, b = case
    print(f"{name:34} nums1={a} nums2={b}: sai -> {fn(a, b)}, đúng -> {lines_brute(a, b)}")
    check_all(a, b)
print("12.3 phản ví dụ: OK")

tham lam nối ngay                  nums1=[1, 2, 1] nums2=[2, 1]: sai -> 1, đúng -> 2
thiếu max khi khác nhau            nums1=[1] nums2=[1, 2]: sai -> 0, đúng -> 1
nhầm với chuỗi con liên tiếp       nums1=[1, 1] nums2=[1, 2, 1]: sai -> 1, đúng -> 2
đếm giá trị chung (tập hợp)        nums1=[1, 1] nums2=[1, 1]: sai -> 1, đúng -> 2
mảng cuộn quên ô chéo              nums1=[1] nums2=[1, 1]: sai -> 2, đúng -> 1
12.3 phản ví dụ: OK


In [22]:
# 12.4 ngẫu nhiên: nhiều miền giá trị, kích thước nhỏ so oracle; kích thước lớn so các cách nhanh
rng = random.Random(4035)
for _ in range(600):
    la, lb = rng.randint(1, 7), rng.randint(1, 7)
    vmax = rng.choice([2, 3, 5, 2000])
    check_all([rng.randint(1, vmax) for _ in range(la)], [rng.randint(1, vmax) for _ in range(lb)])
for _ in range(10):
    la, lb = rng.randint(100, 500), rng.randint(100, 500)
    vmax = rng.choice([2, 5, 50, 2000])
    a, b = [rng.randint(1, vmax) for _ in range(la)], [rng.randint(1, vmax) for _ in range(lb)]
    check_all(a, b, brute=False)
print("12.4 ngẫu nhiên: OK")

# 12.5 tính chất: đối xứng khi đổi hai mảng; <= min độ dài; = độ dài khi mảng này là dãy con của mảng kia; đảo cả hai mảng không đổi đáp án; thêm phần tử không làm giảm
rng = random.Random(5035)
for _ in range(300):
    la, lb = rng.randint(1, 30), rng.randint(1, 30)
    a = [rng.randint(1, 4) for _ in range(la)]
    b = [rng.randint(1, 4) for _ in range(lb)]
    r = sol.maxUncrossedLines(a, b)
    assert r == sol.maxUncrossedLines(b, a) <= min(la, lb)
    assert r == sol.maxUncrossedLines(a[::-1], b[::-1])
    assert sol.maxUncrossedLines(a + [rng.randint(1, 4)], b) >= r
    keep = [x for x in a if rng.random() < 0.6] or [a[0]]
    assert sol.maxUncrossedLines(a, keep) == len(keep)             # keep là dãy con của a
print("12.5 tính chất: OK")

12.4 ngẫu nhiên: OK
12.5 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: nối ngay các cặp bằng nhau đầu tiên

Nối `nums1[i]` với vị trí khớp **đầu tiên** của `nums2` có thể chặn đường các đoạn nối tốt hơn về sau.

In [23]:
a, b = counters["tham lam nối ngay"]
print(f"nums1={a} nums2={b}: sai ->", wrong_greedy(a, b), "| đúng ->", Solution().maxUncrossedLines(a, b))
assert wrong_greedy(a, b) != Solution().maxUncrossedLines(a, b)

nums1=[1, 2, 1] nums2=[2, 1]: sai -> 1 | đúng -> 2


Vì sao sai: nối vội làm "dùng hết" phần đầu của `nums2` nên mất các cặp phù hợp hơn. **Đúng:** quy hoạch động xét mọi khả năng.

### Lỗi 2 — Thiếu `max` khi hai số khác nhau

Khi `nums1[i-1] != nums2[j-1]` phải lấy `max(dp[i-1][j], dp[i][j-1])`; chỉ lấy ô chéo làm mất các đoạn nối đã có.

In [24]:
a, b = counters["thiếu max khi khác nhau"]
print(f"nums1={a} nums2={b}: sai ->", wrong_no_max(a, b), "| đúng ->", Solution().maxUncrossedLines(a, b))
assert wrong_no_max(a, b) != Solution().maxUncrossedLines(a, b)

nums1=[1] nums2=[1, 2]: sai -> 0 | đúng -> 1


Vì sao sai: bỏ `nums1[i-1]` (hoặc `nums2[j-1]`) vẫn có thể giữ các đoạn nối tốt từ các tiền tố dài hơn. **Đúng:** `max(dp[i-1][j], dp[i][j-1])`.

### Lỗi 3 — Nhầm với chuỗi con **liên tiếp** (substring)

Reset về `0` khi khác nhau chỉ tính các đoạn liên tiếp; bài này cho phép **bỏ qua** phần tử (subsequence).

In [25]:
a, b = counters["nhầm với chuỗi con liên tiếp"]
print(f"nums1={a} nums2={b}: sai ->", wrong_substring(a, b), "| đúng ->", Solution().maxUncrossedLines(a, b))
assert wrong_substring(a, b) != Solution().maxUncrossedLines(a, b)
print("ví dụ 2:", wrong_substring(*examples[1][:2]), "(sai) so với", Solution().maxUncrossedLines(*examples[1][:2]))

nums1=[1, 1] nums2=[1, 2, 1]: sai -> 1 | đúng -> 2
ví dụ 2: 1 (sai) so với 3


Vì sao sai: các số được nối không cần nằm liền nhau trong mảng. **Đúng:** giữ giá trị `max` của hai ô (trên, trái) thay vì reset.

### Lỗi 4 — Đếm số giá trị chung (tập hợp)

Giá trị chung không bảo đảm vẽ được nhiều đoạn không cắt: thứ tự và số lần xuất hiện quan trọng.

In [26]:
a, b = counters["đếm giá trị chung (tập hợp)"]
print(f"nums1={a} nums2={b}: sai ->", wrong_set_intersection(a, b), "| đúng ->", Solution().maxUncrossedLines(a, b))
assert wrong_set_intersection(a, b) != Solution().maxUncrossedLines(a, b)
print("ví dụ 1:", wrong_set_intersection([1, 4, 2], [1, 2, 4]), "(sai: 3 giá trị chung) so với", Solution().maxUncrossedLines([1, 4, 2], [1, 2, 4]))

nums1=[1, 1] nums2=[1, 1]: sai -> 1 | đúng -> 2
ví dụ 1: 3 (sai: 3 giá trị chung) so với 2


Vì sao sai: ví dụ 1 có 3 giá trị chung nhưng chỉ vẽ được 2 đoạn không cắt. **Đúng:** LCS.

### Lỗi 5 — Mảng cuộn: ghi đè ô chéo

Khi cập nhật tại chỗ, `dp[j-1]` đã là giá trị **hàng mới** (không phải ô chéo). Cần cất ô trên cũ vào `prev` trước khi ghi.

In [27]:
a, b = counters["mảng cuộn quên ô chéo"]
print(f"nums1={a} nums2={b}: sai ->", wrong_rolling_no_prev(a, b), "| đúng ->", Solution().maxUncrossedLines(a, b))
assert wrong_rolling_no_prev(a, b) != Solution().maxUncrossedLines(a, b)

nums1=[1] nums2=[1, 1]: sai -> 2 | đúng -> 1


Vì sao sai: dùng `dp[j - 1] + 1` có thể đếm cùng một phần tử của `nums1` hai lần. **Đúng:** biến `prev` lưu `dp[i-1][j-1]` trước khi bị ghi đè.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "nối cặp bằng nhau không cắt nhau", "giữ thứ tự", "dãy con chung" ⇒ **LCS**, quy hoạch động theo hai tiền tố `O(n * m)`.
- Công thức: bằng nhau ⇒ `dp[i-1][j-1] + 1`; khác nhau ⇒ `max(dp[i-1][j], dp[i][j-1])`.
- Bảng kích thước `(n + 1) x (m + 1)` với hàng/cột `0` là `0` để tránh xét biên.
- Có thể rút gọn bộ nhớ bằng **mảng cuộn** (nhớ cất ô chéo), hoặc đổi sang bài LIS khi các giá trị thưa.
- Biến thể "xóa tối thiểu để hai chuỗi bằng nhau" `= n + m - 2 * LCS` (bài 583).
- Cùng họ: Longest Common Subsequence (1143), Delete Operation for Two Strings (583), Edit Distance (72), Shortest Common Supersequence (1092).
- Test nên có: không có giá trị chung, hai mảng giống hệt, một mảng đảo ngược, toàn số giống nhau, `500 x 500`.

## 15. Biến thể

1. **Xóa tối thiểu để hai mảng bằng nhau** (583): `n + m - 2 * LCS`.
2. **Dựng một dãy đoạn nối cụ thể** (truy vết bảng `dp`): bài tập 5.
3. **Ba mảng** (LCS của ba mảng): bảng 3 chiều.
4. Bài liên quan: 1143, 583, 72.

Biến thể 1 kiểm bằng oracle quy hoạch động độc lập về xóa phần tử:

In [28]:
def min_deletions_to_equal(a: List[int], b: List[int]) -> int:
    return len(a) + len(b) - 2 * Solution().maxUncrossedLines(a, b)

def min_deletions_brute(a: List[int], b: List[int]) -> int:
    n, m = len(a), len(b)
    INF = 10 ** 9
    f = [[INF] * (m + 1) for _ in range(n + 1)]
    f[0][0] = 0
    for i in range(n + 1):
        for j in range(m + 1):
            if i < n:
                f[i + 1][j] = min(f[i + 1][j], f[i][j] + 1)
            if j < m:
                f[i][j + 1] = min(f[i][j + 1], f[i][j] + 1)
            if i < n and j < m and a[i] == b[j]:
                f[i + 1][j + 1] = min(f[i + 1][j + 1], f[i][j])
    return f[n][m]

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]
    assert min_deletions_to_equal(a, b) == min_deletions_brute(a, b)
print("biến thể xóa tối thiểu = n + m - 2 * LCS: khớp DP xóa độc lập")

biến thể xóa tối thiểu = n + m - 2 * LCS: khớp DP xóa độc lập


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập đoạn nối | Theo định nghĩa | hàm mũ | `O(n + m)` | Oracle |
| 2. Đệ quy có nhớ | Top-down | `O(nm)` | `O(nm)` | Dễ suy ra từ công thức |
| 3. Bảng 2 chiều | Bottom-up | `O(nm)` | `O(nm)` | Cần truy vết |
| 4. Mảng cuộn | Một hàng | `O(nm)` | `O(min(n, m))` | **Cách nộp chính** |
| 5. LIS qua vị trí | `bisect` | `O(R log R)` | `O(m)` | Giá trị thưa |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(compatible((0, 0), (1, 1)), compatible((0, 1), (1, 0)), compatible((2, 2), (2, 3)))
print(Solution().maxUncrossedLines([1, 2], [2, 1]), Solution().maxUncrossedLines([3, 3], [3]))
```

In [29]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba bool
my_guess_2 = None      # hai số

### Bài 2 — Tính tay (dễ-vừa)

`nums1 = [1, 3, 2, 1]`, `nums2 = [3, 1, 2]`. Ghi đáp án.

In [30]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành bảng 2 chiều.

In [31]:
def exercise_3(nums1: List[int], nums2: List[int]) -> int:
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # TODO 1: nếu nums1[i - 1] == nums2[j - 1]: dp[i][j] = dp[i - 1][j - 1] + 1
            # TODO 2: ngược lại: dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
            pass
    # TODO 3: trả về dp[n][m]
    pass


# for a, b, expected in examples:
#     assert exercise_3(a, b) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, tự viết `exercise_4(nums1, nums2)` bằng **LIS** trên danh sách vị trí (duyệt các vị trí của mỗi `x` theo thứ tự **giảm**).

In [32]:
def exercise_4(nums1: List[int], nums2: List[int]) -> int:
    # TODO
    pass


# for a, b, expected in examples:
#     assert exercise_4(a, b) == expected

### Bài 5 — Mini challenge: dựng các đoạn nối (khó)

Viết `exercise_5(nums1, nums2)` trả về **một** danh sách các đoạn nối `(i, j)` (chỉ số trong `nums1`, `nums2`) đạt số đoạn tối đa, không cắt nhau. Có nhiều đáp án đúng; ta kiểm tra bằng **hàm kiểm tra**: mỗi đoạn nối hai số bằng nhau, các đoạn đôi một tương thích, và số đoạn bằng đáp án.

```python
exercise_5([1, 4, 2], [1, 2, 4])   -> ví dụ [(0, 0), (1, 2)] hoặc [(0, 0), (2, 1)]
exercise_5([1, 2], [3, 4])         -> []
```
Gợi ý: truy vết bảng `dp` từ `(n, m)` về `(0, 0)`.

In [33]:
def exercise_5(nums1: List[int], nums2: List[int]) -> List[Tuple[int, int]]:
    # TODO
    pass


def check_lines(nums1, nums2, lines) -> bool:
    return (all(0 <= i < len(nums1) and 0 <= j < len(nums2) and nums1[i] == nums2[j] for i, j in lines)
            and all(compatible(l1, l2) for k, l1 in enumerate(lines) for l2 in lines[k + 1:])
            and len(lines) == Solution().maxUncrossedLines(nums1, nums2))


# for a, b, _ in examples:
#     assert check_lines(a, b, exercise_5(a, b))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: tương thích khi `(i1 - i2) * (j1 - j2) > 0`.
- Mức 2: `(0, 0)`, `(1, 1)`: tích `1 > 0`; `(0, 1)`, `(1, 0)`: tích `-1`; `(2, 2)`, `(2, 3)`: tích `0` (chung đầu mút).
- Mức 3: dòng 1: `True False False`; dòng 2: `1 1`.

**Bài 2**
- Mức 1: cặp có thể nối: `3-3`, `1-1` (hai vị trí), `2-2`.
- Mức 2: `3` ở `(1, 0)`, `2` ở `(2, 2)`: tương thích (`(1-2)*(0-2) > 0`); `1` ở `(0, 1)` hoặc `(3, 1)`.
- Mức 3: `[3, 2]` hoặc `[1, 2]` đều độ dài `2`; ba cặp không đủ vì `3, 1, 2` không là dãy con của `nums1`; đáp án `2`.

**Bài 3**
- Mức 1: `if nums1[i - 1] == nums2[j - 1]: dp[i][j] = dp[i - 1][j - 1] + 1`.
- Mức 2: `else: dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])`.
- Mức 3: `return dp[n][m]`.

**Bài 4**
- Mức 1: `pos[y]` là danh sách vị trí tăng dần của `y` trong `nums2`.
- Mức 2: với mỗi `x` trong `nums1`, duyệt `reversed(pos[x])`, dùng `bisect_left(tails, j)`.
- Mức 3: `tails[k] = j` (hoặc `append` nếu `k == len(tails)`); trả `len(tails)`.

**Bài 5**
- Mức 1: dựng bảng `dp` đầy đủ (Cách 3).
- Mức 2: từ `(i, j) = (n, m)`: nếu hai số bằng nhau thì ghi `(i - 1, j - 1)` và đi chéo; ngược lại đi theo ô lớn hơn.
- Mức 3: đảo danh sách cuối cùng.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [34]:
print(compatible((0, 0), (1, 1)), compatible((0, 1), (1, 0)), compatible((2, 2), (2, 3)))
print(Solution().maxUncrossedLines([1, 2], [2, 1]), Solution().maxUncrossedLines([3, 3], [3]))

assert (compatible((0, 0), (1, 1)), compatible((0, 1), (1, 0)), compatible((2, 2), (2, 3))) == (True, False, False)
assert (Solution().maxUncrossedLines([1, 2], [2, 1]), Solution().maxUncrossedLines([3, 3], [3])) == (1, 1)

True False False
1 1


### Đáp án Bài 2

Các cặp bằng nhau: `3` (vị trí `(1, 0)`), `1` (`(0, 1)`, `(3, 1)`), `2` (`(2, 2)`). Hai đoạn `(1, 0)` và `(2, 2)` tương thích; hai đoạn `(0, 1)` và `(2, 2)` cũng tương thích. Ba đoạn đồng thời không được (`3, 1, 2` không là dãy con chung). Đáp án `2`.

In [35]:
hand_answer_solution = 2
assert Solution().maxUncrossedLines([1, 3, 2, 1], [3, 1, 2]) == hand_answer_solution == lines_brute([1, 3, 2, 1], [3, 1, 2])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [36]:
def exercise_3_solution(nums1: List[int], nums2: List[int]) -> int:
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if nums1[i - 1] == nums2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a, b, expected in examples:
    assert exercise_3_solution(a, b) == expected
rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    assert exercise_3_solution(a, b) == lines_brute(a, b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def exercise_4_solution(nums1: List[int], nums2: List[int]) -> int:
    pos = defaultdict(list)
    for j, y in enumerate(nums2):
        pos[y].append(j)
    tails = []
    for x in nums1:
        for j in reversed(pos.get(x, ())):
            k = bisect_left(tails, j)
            if k == len(tails):
                tails.append(j)
            else:
                tails[k] = j
    return len(tails)

for a, b, expected in examples:
    assert exercise_4_solution(a, b) == expected
rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    assert exercise_4_solution(a, b) == lines_brute(a, b)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def exercise_5_solution(nums1: List[int], nums2: List[int]) -> List[Tuple[int, int]]:
    n, m = len(nums1), len(nums2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if nums1[i - 1] == nums2[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
    lines = []
    i, j = n, m
    while i > 0 and j > 0:
        if nums1[i - 1] == nums2[j - 1]:
            lines.append((i - 1, j - 1))
            i, j = i - 1, j - 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return lines[::-1]

for a, b, _ in examples:
    assert check_lines(a, b, exercise_5_solution(a, b))
assert exercise_5_solution([1, 2], [3, 4]) == []
rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 10))]
    assert check_lines(a, b, exercise_5_solution(a, b)), (a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (LCS, mảng cuộn)

```python
class Solution:
    def maxUncrossedLines(self, nums1: List[int], nums2: List[int]) -> int:
        if len(nums2) > len(nums1):              # chiều cột là mảng ngắn hơn
            nums1, nums2 = nums2, nums1
        m = len(nums2)
        dp = [0] * (m + 1)
        for x in nums1:
            prev = 0                             # ô chéo dp[i-1][j-1]
            for j in range(1, m + 1):
                tmp = dp[j]                      # ô trên cũ -> thành ô chéo cho j + 1
                if x == nums2[j - 1]:
                    dp[j] = prev + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev = tmp
        return dp[m]
```

Công thức nhớ nhanh: đáp án `= LCS(nums1, nums2)`; `dp[i][j] = dp[i-1][j-1] + 1` nếu bằng nhau, ngược lại `max(dp[i-1][j], dp[i][j-1])`.

### Checklist trước khi submit

- [ ] Nhận ra đáp án là **dãy con chung dài nhất** (không phải chuỗi con liên tiếp, không phải số giá trị chung)?
- [ ] Bảng kích thước `(n + 1) x (m + 1)`, hàng và cột `0` bằng `0`?
- [ ] Khi bằng nhau lấy **ô chéo** `+ 1`; khi khác nhau lấy `max` của ô trên và ô trái?
- [ ] Nếu dùng mảng cuộn: đã cất ô chéo (`prev`) trước khi ghi đè?
- [ ] Nếu dùng đệ quy: nâng giới hạn đệ quy (độ sâu tới `n + m = 1000`)?
- [ ] Đã thử: không có giá trị chung, hai mảng giống hệt, đảo ngược, toàn số giống nhau, `500 x 500`?